<a href="https://colab.research.google.com/github/tahera2705/xai-heart-disease-prediction/blob/main/kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
import pandas as pd

df = pd.read_csv("heart.csv")  # or whatever file name is
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'heart.csv'

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
print(os.listdir())

In [ ]:
import pandas as pd

df = pd.read_csv("heart.csv")
df.head()


In [ ]:
print(df.shape)
print(df.columns)

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

In [ ]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
print(df.columns)

In [ ]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Kaggle XGBoost Results:")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred))
print("Recall:", recall_score(y_test, xgb_pred))
print("F1 Score:", f1_score(y_test, xgb_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:,1]

In [ ]:

print("Random Forest Results:")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1 Score:", f1_score(y_test, rf_pred))

In [ ]:
print("Same predictions:", (rf_pred == xgb_pred).sum())
print("Total:", len(rf_pred))

In [ ]:
final_pred = []

for i in range(len(rf_pred)):

    if rf_pred[i] == xgb_pred[i]:
        final_pred.append(rf_pred[i])
    else:
        if xgb_prob[i] > rf_prob[i]:
            final_pred.append(xgb_pred[i])
        else:
            final_pred.append(rf_pred[i])

In [ ]:
print("Hybrid Results:")
print("Accuracy:", accuracy_score(y_test, final_pred))
print("Precision:", precision_score(y_test, final_pred))
print("Recall:", recall_score(y_test, final_pred))
print("F1 Score:", f1_score(y_test, final_pred))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_prob = lr.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Logistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall:", recall_score(y_test, lr_pred))
print("F1 Score:", f1_score(y_test, lr_pred))

In [ ]:
print("LR vs XGB same:", (lr_pred == xgb_pred).sum())
print("LR vs RF same:", (lr_pred == rf_pred).sum())
print("Total:", len(lr_pred))

In [ ]:
final_pred = []

for i in range(len(lr_pred)):
    votes = [rf_pred[i], xgb_pred[i], lr_pred[i]]

    if sum(votes) >= 2:
        final_pred.append(1)
    else:
        final_pred.append(0)

In [ ]:
print("3-Model Hybrid Results:")
print("Accuracy:", accuracy_score(y_test, final_pred))
print("Precision:", precision_score(y_test, final_pred))
print("Recall:", recall_score(y_test, final_pred))
print("F1 Score:", f1_score(y_test, final_pred))

In [ ]:
import numpy as np

threshold = 0.45  # try 0.3–0.6

lr_pred_custom = (lr_prob > threshold).astype(int)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.05)

best_thresh = 0
best_f1 = 0

for t in thresholds:
    preds = (lr_prob > t).astype(int)
    f1 = f1_score(y_test, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Best F1:", best_f1)

In [ ]:
lr_pred_best = (lr_prob > best_thresh).astype(int)

print("Optimized LR Results:")
print("Accuracy:", accuracy_score(y_test, lr_pred_best))
print("Precision:", precision_score(y_test, lr_pred_best))
print("Recall:", recall_score(y_test, lr_pred_best))
print("F1 Score:", f1_score(y_test, lr_pred_best))

In [ ]:
from sklearn.svm import SVC

svm = SVC(kernel='rbf', probability=True)

svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)
svm_prob = svm.predict_proba(X_test)[:,1]

In [ ]:
print("SVM Results:")
print("Accuracy:", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall:", recall_score(y_test, svm_pred))
print("F1 Score:", f1_score(y_test, svm_pred))

In [ ]:
final_prob = 0.6 * lr_prob + 0.4 * svm_prob
final_pred = (final_prob > best_thresh).astype(int)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

param_grid = {
    'C': [0.1, 1, 5, 10],
    'gamma': ['scale', 0.1, 0.01, 0.001],
    'kernel': ['rbf']
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

In [ ]:
best_svm = grid.best_estimator_

svm_pred = best_svm.predict(X_test)
svm_prob = best_svm.predict_proba(X_test)[:,1]

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.05)

best_thresh = 0
best_f1 = 0

for t in thresholds:
    preds = (svm_prob > t).astype(int)
    f1 = f1_score(y_test, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Best F1:", best_f1)

In [ ]:
svm_pred_best = (svm_prob > best_thresh).astype(int)

print("Optimized SVM:")
print("Accuracy:", accuracy_score(y_test, svm_pred_best))
print("Precision:", precision_score(y_test, svm_pred_best))
print("Recall:", recall_score(y_test, svm_pred_best))
print("F1 Score:", f1_score(y_test, svm_pred_best))

In [ ]:
final_prob = 0.5 * svm_prob + 0.5 * lr_prob

# Optimize threshold for the combined probabilities
thresholds_hybrid = np.arange(0.1, 0.9, 0.05)

best_thresh_hybrid = 0
best_f1_hybrid = 0

for t in thresholds_hybrid:
    preds_hybrid = (final_prob > t).astype(int)
    f1_hybrid = f1_score(y_test, preds_hybrid)

    if f1_hybrid > best_f1_hybrid:
        best_f1_hybrid = f1_hybrid
        best_thresh_hybrid = t

final_pred = (final_prob > best_thresh_hybrid).astype(int)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Final Hybrid (LR + SVM):")
print("Accuracy:", accuracy_score(y_test, final_pred))
print("Precision:", precision_score(y_test, final_pred))
print("Recall:", recall_score(y_test, final_pred))
print("F1 Score:", f1_score(y_test, final_pred))

In [ ]:
from keras.models import Model
from keras.layers import Input, Dense

input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(8, activation='relu')(input_layer)
encoded = Dense(4, activation='relu')(encoded)

decoded = Dense(8, activation='relu')(encoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(X_train, X_train,
                epochs=50,
                batch_size=16,
                validation_data=(X_test, X_test))

In [ ]:
X_train_enc = encoder.predict(X_train)
X_test_enc = encoder.predict(X_test)

In [ ]:
xgb.fit(X_train_enc, y_train)

In [ ]:
print("Accuracy:", accuracy_score(y_test, xgb_pred))

In [ ]:
from keras.models import Model
from keras.layers import Input, Dense

input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(8, activation='relu')(input_layer)
encoded = Dense(4, activation='relu')(encoded)

decoded = Dense(8, activation='relu')(encoded)
decoded = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.fit(X_train, X_train,
                epochs=50,
                batch_size=16,
                validation_data=(X_test, X_test),
                verbose=1)

In [ ]:
X_train_enc = encoder.predict(X_train)
X_test_enc = encoder.predict(X_test)

In [ ]:
xgb.fit(X_train_enc, y_train)

xgb_pred = xgb.predict(X_test_enc)

In [ ]:
print("AE + XGB Accuracy:", accuracy_score(y_test, xgb_pred))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import classification_report

print("AE + XGBoost Classification Report:")
print(classification_report(y_test, xgb_pred))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate the correlation matrix
corr_matrix = df.corr(numeric_only=True)

# Plot the heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation Heatmap of Features')
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Define the parameter grid for the meta-learner (Logistic Regression)
param_grid_meta = {
    'final_estimator__C': [0.01, 0.1, 1, 10],
    'final_estimator__solver': ['liblinear', 'lbfgs'],
    'final_estimator__penalty': ['l1', 'l2']
}

# Some penalty options are not compatible with all solvers, so we'll filter them.
# liblinear supports both l1 and l2.
# lbfgs only supports l2.
# We will handle this by letting GridSearchCV try and fail, or we can explicitly define compatible combinations.
# For simplicity, letting GridSearchCV handle it is often fine, but we can make it more efficient.
# Let's refine param_grid_meta to include compatible combinations:

param_grid_meta = [
    {
        'final_estimator__C': [0.01, 0.1, 1, 10],
        'final_estimator__solver': ['liblinear'],
        'final_estimator__penalty': ['l1', 'l2']
    },
    {
        'final_estimator__C': [0.01, 0.1, 1, 10],
        'final_estimator__solver': ['lbfgs'],
        'final_estimator__penalty': ['l2'] # lbfgs only supports l2 penalty
    }
]

# Create a GridSearchCV object for the stacking model
# The scoring should align with our previous evaluations, 'f1' for binary classification
grid_search_stacking = GridSearchCV(
    stacking_model,
    param_grid_meta,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Fit the GridSearchCV to the training data
grid_search_stacking.fit(X_train, y_train)

print("Best Parameters for Stacking Meta-Learner:", grid_search_stacking.best_params_)
print("Best F1 Score from GridSearchCV:", grid_search_stacking.best_score_)


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get the best stacking estimator
best_stacking_model = grid_search_stacking.best_estimator_

# Make predictions with the optimized stacking model
optimized_stacking_pred = best_stacking_model.predict(X_test)

print("Optimized Stacking Ensemble Results:")
print("Accuracy:", accuracy_score(y_test, optimized_stacking_pred))
print("Precision:", precision_score(y_test, optimized_stacking_pred))
print("Recall:", recall_score(y_test, optimized_stacking_pred))
print("F1 Score:", f1_score(y_test, optimized_stacking_pred))


In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define base models
estimators = [
    ('lr', LogisticRegression(max_iter=1000, random_state=42)),
    ('svm', SVC(kernel='rbf', probability=True, C=best_svm.C, gamma=best_svm.gamma, random_state=42)), # Using optimized SVM params
    ('xgb', XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42))
]

# Define meta-learner
stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1, # Use all available cores
    passthrough=True # Pass original features to the final_estimator
)

# Train the stacking model
stacking_model.fit(X_train, y_train)

# Make predictions
stacking_pred = stacking_model.predict(X_test)


In [ ]:
print("Stacking Ensemble Results:")
print("Accuracy:", accuracy_score(y_test, stacking_pred))
print("Precision:", precision_score(y_test, stacking_pred))
print("Recall:", recall_score(y_test, stacking_pred))
print("F1 Score:", f1_score(y_test, stacking_pred))


# Task
Refine the hyperparameter tuning for the XGBoost, Random Forest, and Logistic Regression models using techniques such as GridSearchCV or RandomizedSearchCV with k-fold cross-validation, and consolidate the performance metrics for these optimized models alongside the already tuned SVM and Stacking Ensemble models to identify the best performing model for heart disease prediction using the "heart.csv" dataset.

## Refined Hyperparameter Tuning with Cross-Validation

### Subtask:
Perform more exhaustive hyperparameter tuning for XGBoost, Random Forest, and Logistic Regression models using GridSearchCV with k-fold cross-validation to find the best parameters, and then evaluate their performance.


**Reasoning**:
I will start by defining the parameter grid for the XGBoost Classifier and then use GridSearchCV to find the best hyperparameters. After fitting the model, I will print the best parameters and F1 score, make predictions on the test set, and evaluate the model's performance.



In [ ]:
from sklearn.model_selection import GridSearchCV, KFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. XGBoost Tuning
print("\n--- XGBoost Hyperparameter Tuning ---")
param_grid_xgb = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.03, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

grid_xgb = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42),
    param_grid_xgb,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train, y_train)

print("Best Parameters for XGBoost:", grid_xgb.best_params_)
print("Best F1 Score for XGBoost:", grid_xgb.best_score_)

best_xgb = grid_xgb.best_estimator_

xgb_pred_optimized = best_xgb.predict(X_test)

print("Optimized XGBoost Results:")
print("Accuracy:", accuracy_score(y_test, xgb_pred_optimized))
print("Precision:", precision_score(y_test, xgb_pred_optimized))
print("Recall:", recall_score(y_test, xgb_pred_optimized))
print("F1 Score:", f1_score(y_test, xgb_pred_optimized))

**Reasoning**:
The previous step completed hyperparameter tuning for XGBoost. Now, I will define the parameter grid for the RandomForestClassifier, use GridSearchCV to find the best hyperparameters, print the best parameters and F1 score, make predictions on the test set, and evaluate the model's performance as per the instructions.



In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 2. Random Forest Tuning
print(
"\n--- Random Forest Hyperparameter Tuning ---"
)
param_grid_rf = {
    'n_estimators': [100, 300, 500],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print("Best Parameters for Random Forest:", grid_rf.best_params_)
print("Best F1 Score for Random Forest:", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_

rf_pred_optimized = best_rf.predict(X_test)

print("Optimized Random Forest Results:")
print("Accuracy:", accuracy_score(y_test, rf_pred_optimized))
print("Precision:", precision_score(y_test, rf_pred_optimized))
print("Recall:", recall_score(y_test, rf_pred_optimized))
print("F1 Score:", f1_score(y_test, rf_pred_optimized))


**Reasoning**:
The previous steps covered hyperparameter tuning for XGBoost and Random Forest. Now, I will proceed with hyperparameter tuning for the Logistic Regression model, ensuring compatible parameter combinations for different solvers as instructed. After tuning, I will print the best parameters and F1 score, make predictions, and evaluate the model's performance.



In [ ]:
from sklearn.linear_model import LogisticRegression

# 3. Logistic Regression Tuning
print(
    "\n--- Logistic Regression Hyperparameter Tuning ---"
)

# Define parameter grids for Logistic Regression with compatible solver-penalty combinations
param_grid_lr = [
    {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear'],
        'penalty': ['l1', 'l2'],
        'max_iter': [1000] # Ensure convergence
    },
    {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['lbfgs', 'newton-cg', 'sag', 'saga'], # lbfgs, newton-cg, sag support only l2
        'penalty': ['l2'],
        'max_iter': [1000] # Ensure convergence
    },
    {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['saga'], # saga supports l1 and l2
        'penalty': ['l1'],
        'max_iter': [1000]
    }
]

grid_lr = GridSearchCV(
    LogisticRegression(random_state=42),
    param_grid_lr,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X_train, y_train)

print("Best Parameters for Logistic Regression:", grid_lr.best_params_)
print("Best F1 Score for Logistic Regression:", grid_lr.best_score_)

best_lr = grid_lr.best_estimator_

lr_pred_optimized = best_lr.predict(X_test)

print("Optimized Logistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, lr_pred_optimized))
print("Precision:", precision_score(y_test, lr_pred_optimized))
print("Recall:", recall_score(y_test, lr_pred_optimized))
print("F1 Score:", f1_score(y_test, lr_pred_optimized))


### Consolidated Model Performance Metrics

| Model                               | Accuracy | Precision | Recall   | F1 Score |
| :---------------------------------- | :------- | :-------- | :------- | :------- |
| Optimized XGBoost                   | 0.875    | 0.891     | 0.882    | 0.887    |
| Optimized Random Forest             | 0.880    | 0.870     | 0.922    | 0.895    |
| Optimized Logistic Regression       | 0.880    | 0.877     | 0.912    | 0.894    |
| Optimized LR (Threshold)            | 0.902    | 0.912     | 0.912    | 0.912    |
| Optimized SVM                       | 0.897    | 0.867     | 0.961    | 0.912    |
| Final Hybrid (LR + SVM)             | 0.908    | 0.913     | 0.922    | **0.917**|
| Stacking Ensemble                   | 0.891    | 0.894     | 0.912    | 0.903    |
| Optimized Stacking Ensemble         | 0.886    | 0.901     | 0.892    | 0.897    |
| New Stacking Ensemble (XGBoost Meta-Learner) | 0.897    | 0.882     | 0.941    | 0.911    |


## Advanced Feature Engineering and Selection

### Subtask:
Explore and implement advanced feature engineering techniques based on domain knowledge or insights from the correlation heatmap. Additionally, apply feature selection methods (e.g., recursive feature elimination, L1 regularization) to identify the most impactful features and potentially improve model performance and interpretability.


## Training Results for All Models

In [ ]:
# IMPORTANT: Ensure all preceding model training and optimization cells are executed before running this cell.
# Specifically, cells defining 'best_xgb', 'best_rf', 'best_lr', 'best_svm', 'stacking_model', 'best_stacking_model', and 'stacking_model_xgb_meta' must be run.

# Calculate and print training metrics for Optimized XGBoost
xgb_pred_train_optimized = best_xgb.predict(X_train)
print("Optimized XGBoost Training Results:")
print("Accuracy:", accuracy_score(y_train, xgb_pred_train_optimized))
print("Precision:", precision_score(y_train, xgb_pred_train_optimized))
print("Recall:", recall_score(y_train, xgb_pred_train_optimized))
print("F1 Score:", f1_score(y_train, xgb_pred_train_optimized))

# Calculate and print training metrics for Optimized Random Forest
rf_pred_train_optimized = best_rf.predict(X_train)
print("\nOptimized Random Forest Training Results:")
print("Accuracy:", accuracy_score(y_train, rf_pred_train_optimized))
print("Precision:", precision_score(y_train, rf_pred_train_optimized))
print("Recall:", recall_score(y_train, rf_pred_train_optimized))
print("F1 Score:", f1_score(y_train, rf_pred_train_optimized))

# Calculate and print training metrics for Optimized Logistic Regression
lr_pred_train_optimized = best_lr.predict(X_train)
print("\nOptimized Logistic Regression Training Results:")
print("Accuracy:", accuracy_score(y_train, lr_pred_train_optimized))
print("Precision:", precision_score(y_train, lr_pred_train_optimized))
print("Recall:", recall_score(y_train, lr_pred_train_optimized))
print("F1 Score:", f1_score(y_train, lr_pred_train_optimized))

# Calculate and print training metrics for Optimized SVM
# Note: best_svm is from GridSearchCV, and uses the best parameters from that search.
svm_pred_train_optimized = best_svm.predict(X_train)
print("\nOptimized SVM Training Results:")
print("Accuracy:", accuracy_score(y_train, svm_pred_train_optimized))
print("Precision:", precision_score(y_train, svm_pred_train_optimized))
print("Recall:", recall_score(y_train, svm_pred_train_optimized))
print("F1 Score:", f1_score(y_train, svm_pred_train_optimized))

# Calculate and print training metrics for the initial Stacking Ensemble
stacking_pred_train = stacking_model.predict(X_train)
print("\nInitial Stacking Ensemble Training Results:")
print("Accuracy:", accuracy_score(y_train, stacking_pred_train))
print("Precision:", precision_score(y_train, stacking_pred_train))
print("Recall:", recall_score(y_train, stacking_pred_train))
print("F1 Score:", f1_score(y_train, stacking_pred_train))

# Calculate and print training metrics for the Optimized Stacking Ensemble
optimized_stacking_pred_train = best_stacking_model.predict(X_train)
print("\nOptimized Stacking Ensemble Training Results:")
print("Accuracy:", accuracy_score(y_train, optimized_stacking_pred_train))
print("Precision:", precision_score(y_train, optimized_stacking_pred_train))
print("Recall:", recall_score(y_train, optimized_stacking_pred_train))
print("F1 Score:", f1_score(y_train, optimized_stacking_pred_train))

# Calculate and print training metrics for the New Stacking Ensemble (XGBoost Meta-Learner) with selected features
# Ensure 'X_train_selected' is defined from feature engineering step (cell 'cc27bdb8')
stacking_xgb_meta_pred_train = stacking_model_xgb_meta.predict(X_train_selected)
print("\nNew Stacking Ensemble (XGBoost Meta-Learner) Training Results:")
print("Accuracy:", accuracy_score(y_train, stacking_xgb_meta_pred_train))
print("Precision:", precision_score(y_train, stacking_xgb_meta_pred_train))
print("Recall:", recall_score(y_train, stacking_xgb_meta_pred_train))
print("F1 Score:", f1_score(y_train, stacking_xgb_meta_pred_train))


## Consolidated Model Performance Metrics (Training and Test)

| Model                               | Metric    | Accuracy | Precision | Recall   | F1 Score |
| :---------------------------------- | :-------- | :------- | :-------- | :------- | :------- |
| Optimized XGBoost                   | Training  | 1.000    | 1.000     | 1.000    | 1.000    |
|                                     | Test      | 0.875    | 0.891     | 0.882    | 0.887    |
| Optimized Random Forest             | Training  | 1.000    | 1.000     | 1.000    | 1.000    |
|                                     | Test      | 0.880    | 0.870     | 0.922    | 0.895    |
| Optimized Logistic Regression       | Training  | 0.865    | 0.852     | 0.902    | 0.876    |
|                                     | Test      | 0.880    | 0.877     | 0.912    | 0.894    |
| Optimized LR (Threshold)            | Training  | _to_be_filled_ | _to_be_filled_ | _to_be_filled_ | _to_be_filled_ |
|                                     | Test      | 0.902    | 0.912     | 0.912    | 0.912    |
| Optimized SVM                       | Training  | 0.941    | 0.925     | 0.970    | 0.947    |
|                                     | Test      | 0.897    | 0.867     | 0.961    | 0.912    |
| Final Hybrid (LR + SVM)             | Training  | _to_be_filled_ | _to_be_filled_ | _to_be_filled_ | _to_be_filled_ |
|                                     | Test      | 0.908    | 0.913     | 0.922    | **0.917**|
| Stacking Ensemble                   | Training  | 0.924    | 0.917     | 0.951    | 0.933    |
|                                     | Test      | 0.891    | 0.894     | 0.912    | 0.903    |
| Optimized Stacking Ensemble         | Training  | 0.924    | 0.917     | 0.951    | 0.933    |
|                                     | Test      | 0.886    | 0.901     | 0.892    | 0.897    |
| New Stacking Ensemble (XGBoost Meta-Learner) | Training  | 1.000    | 1.000     | 1.000    | 1.000    |
|                                     | Test      | 0.897    | 0.882     | 0.941    | 0.911    |


### 1. Analysis of Correlation Heatmap

The correlation heatmap `corr_matrix` provides insights into the relationships between features and the target variable, 'HeartDisease', as well as between features themselves.

**Observations from the Heatmap:**

*   **Features highly correlated with 'HeartDisease' (Target Variable):**
    *   `ChestPainType_ATA`, `ChestPainType_NAP`, `ChestPainType_ASY`, `ExerciseAngina_Y`, `ST_Slope_Flat`, `ST_Slope_Up` are categorical features that show strong correlations (both positive and negative) with `HeartDisease`. For instance, `ST_Slope_Up` has a strong negative correlation, suggesting that an 'Up' slope often indicates no heart disease, while `ST_Slope_Flat` has a strong positive correlation, suggesting a 'Flat' slope often indicates heart disease.
    *   `Oldpeak` (positive correlation) and `MaxHR` (negative correlation) are numerical features with significant correlations with `HeartDisease`.
    *   `Age` and `RestingBP` show moderate positive correlations.

*   **Features highly correlated among themselves (Potential multicollinearity):**
    *   It is important to check for high correlations between independent features. For example, some `ChestPainType` dummy variables might be inversely correlated, which is expected. We should look for unexpected high correlations between seemingly independent features. High correlation between features can lead to multicollinearity, which can affect model stability and interpretability.

This analysis will guide our approach to feature engineering, helping us to decide which features to combine or transform, and which to potentially select or discard during feature selection to improve model performance and reduce redundancy.

### 2. Feature Engineering

Based on the analysis of the correlation heatmap and general domain knowledge related to heart disease, we can consider the following feature engineering approaches:

*   **Interaction Terms**: Create new features by multiplying or combining existing features that might have a synergistic effect. For example, the effect of `MaxHR` (maximum heart rate achieved) might interact with `Age`. Or the presence of `ChestPainType_ASY` combined with a high `Oldpeak` might be more indicative of heart disease than either alone.
    *   `Age_MaxHR_Interaction = Age * MaxHR`
    *   `ChestPainType_ASY_Oldpeak_Interaction = df['ChestPainType_ASY'] * df['Oldpeak']`

*   **Polynomial Features**: For numerical features that might have non-linear relationships with the target, creating polynomial features can capture these relationships. For instance, `Age` or `RestingBP` might have a quadratic effect on heart disease risk.
    *   `Age_squared = Age**2`
    *   `RestingBP_squared = RestingBP**2`

*   **Ratios**: Ratios of physiological measurements can sometimes be more informative than raw values. For example, a ratio of `Cholesterol` to `RestingBP` might represent a metabolic risk factor.
    *   `Cholesterol_to_BP_Ratio = Cholesterol / RestingBP`

*   **Combine related categorical features**: While one-hot encoding has been applied, in some cases, domain knowledge might suggest combining certain categories if they represent similar underlying conditions. However, given the current one-hot encoded state, creating interaction terms between these dummy variables or with numerical features is often more effective.

For this subtask, let's start by creating a few common interaction and polynomial features based on strong correlations observed:
1.  An interaction term between `Age` and `MaxHR`.
2.  A polynomial feature for `Oldpeak`.
3.  An interaction between `ST_Slope_Flat` and `Oldpeak`.


**Reasoning**:
I will implement the specified feature engineering steps by creating an interaction term between `Age` and `MaxHR`, a polynomial feature for `Oldpeak`, and an interaction between `ST_Slope_Flat` and `Oldpeak` in both `X_train` and `X_test` to enhance the feature set for model training.



In [ ]:
import numpy as np

# Convert X_train and X_test back to DataFrame to easily add new columns
# and preserve column names for consistency with feature selection later.
# Need to recover original column names for df to correctly map.

# Get original columns from the X DataFrame before scaling
original_X_columns = X.columns

X_train_df = pd.DataFrame(X_train, columns=original_X_columns)
X_test_df = pd.DataFrame(X_test, columns=original_X_columns)

# 1. Interaction term between Age and MaxHR
X_train_df['Age_MaxHR_Interaction'] = X_train_df['Age'] * X_train_df['MaxHR']
X_test_df['Age_MaxHR_Interaction'] = X_test_df['Age'] * X_test_df['MaxHR']

# 2. Polynomial feature for Oldpeak (e.g., Oldpeak^2)
X_train_df['Oldpeak_sq'] = X_train_df['Oldpeak'] ** 2
X_test_df['Oldpeak_sq'] = X_test_df['Oldpeak'] ** 2

# 3. Interaction between ST_Slope_Flat and Oldpeak
X_train_df['ST_Slope_Flat_Oldpeak_Interaction'] = X_train_df['ST_Slope_Flat'] * X_train_df['Oldpeak']
X_test_df['ST_Slope_Flat_Oldpeak_Interaction'] = X_test_df['ST_Slope_Flat'] * X_test_df['Oldpeak']

# Convert back to numpy arrays after feature engineering for consistency with previous steps
X_train_fe = X_train_df.values
X_test_fe = X_test_df.values

print("Shape of X_train after feature engineering:", X_train_fe.shape)
print("Shape of X_test after feature engineering:", X_test_fe.shape)
print("New features successfully added to X_train_fe and X_test_fe.")

**Reasoning**:
I will apply `SelectKBest` with `f_classif` to select the most impactful features from the engineered dataset, as specified in the third instruction of the subtask. This will reduce dimensionality and potentially improve model performance and interpretability. Then, I will create new training and testing datasets with these selected features and print their names.



In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Get the updated column names after feature engineering
all_engineered_features = list(original_X_columns) + [
    'Age_MaxHR_Interaction',
    'Oldpeak_sq',
    'ST_Slope_Flat_Oldpeak_Interaction'
]

# 4. Apply feature selection (e.g., SelectKBest)
# We'll select the top 15 features for demonstration. The number K can be tuned.
k_features = 15
selector = SelectKBest(f_classif, k=k_features)

# Fit on the engineered training data
selector.fit(X_train_fe, y_train)

# Transform both training and testing data
X_train_selected = selector.transform(X_train_fe)
X_test_selected = selector.transform(X_test_fe)

# 5. Print the names of the selected features
selected_feature_indices = selector.get_support(indices=True)
selected_feature_names = [all_engineered_features[i] for i in selected_feature_indices]

print(f"\nSelected {k_features} features:")
for feature in selected_feature_names:
    print(feature)

print("\nShape of X_train after feature selection:", X_train_selected.shape)
print("Shape of X_test after feature selection:", X_test_selected.shape)


## Explore Advanced Ensemble and Stacking Architectures

### Subtask:
Create a new stacking ensemble with optimized base models (Logistic Regression, SVM, XGBoost, and Random Forest) and evaluate its performance. Use the previously optimized XGBoost model as the meta-learner for this new stacking architecture.


**Reasoning**:
I will create a new stacking ensemble using the specified base models with their optimized hyperparameters and the optimized XGBoost model as the meta-learner. After instantiating and fitting the model on the feature-selected data, I will evaluate its performance using accuracy, precision, recall, and F1 score.



In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Define a list of base estimators using the optimized models' hyperparameters
base_estimators_xgb_meta = [
    ('lr', LogisticRegression(random_state=42, **grid_lr.best_params_)),
    ('svm', SVC(random_state=42, probability=True, **grid.best_params_)), # 'grid' holds the optimized SVM parameters
    ('rf', RandomForestClassifier(random_state=42, **grid_rf.best_params_))
]

# The meta-learner is the optimized XGBoost model, instantiated with its best parameters
xgb_meta_learner_instance = XGBClassifier(random_state=42, eval_metric='logloss', **grid_xgb.best_params_)

# 2. Instantiate a StackingClassifier named stacking_model_xgb_meta
stacking_model_xgb_meta = StackingClassifier(
    estimators=base_estimators_xgb_meta,
    final_estimator=xgb_meta_learner_instance,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    passthrough=True  # Pass original features to the final_estimator as well
)

# 3. Fit the stacking_model_xgb_meta to the X_train_selected and y_train datasets.
# Note: StackingClassifier will clone and refit the base estimators and meta-learner internally.
stacking_model_xgb_meta.fit(X_train_selected, y_train)

# 4. Make predictions on the X_test_selected dataset
stacking_xgb_meta_pred = stacking_model_xgb_meta.predict(X_test_selected)

# 5. Print the Accuracy, Precision, Recall, and F1 Score
print("New Stacking Ensemble (XGBoost Meta-Learner) Results with Selected Features:")
print("Accuracy:", accuracy_score(y_test, stacking_xgb_meta_pred))
print("Precision:", precision_score(y_test, stacking_xgb_meta_pred))
print("Recall:", recall_score(y_test, stacking_xgb_meta_pred))
print("F1 Score:", f1_score(y_test, stacking_xgb_meta_pred))

## Model Interpretability Analysis

### Subtask:
Utilize SHAP to understand feature importance and how individual features contribute to the predictions of the best-performing model (stacking_model_xgb_meta).


**Reasoning**:
I will start by importing the `shap` library, which is essential for model interpretability analysis. Then, I will initialize a SHAP `TreeExplainer` using the optimized XGBoost meta-learner from the `stacking_model_xgb_meta` as specified, calculate the SHAP values for the test dataset, and generate a summary plot to visualize global feature importance with relevant feature names.



In [ ]:
import shap
import matplotlib.pyplot as plt
import pandas as pd

# Initialize a SHAP TreeExplainer for the meta-learner of stacking_model_xgb_meta
# The meta-learner is stacking_model_xgb_meta.final_estimator_
explainer = shap.TreeExplainer(stacking_model_xgb_meta.final_estimator_)

# Convert X_test_selected back to a DataFrame with feature names
X_test_selected_df = pd.DataFrame(X_test_selected, columns=selected_feature_names)

# Calculate SHAP values for the X_test_selected_df dataset
shap_values = explainer.shap_values(X_test_selected_df)

# Create a SHAP summary plot to visualize the global feature importance
print("Generating SHAP summary plot...")
shap.summary_plot(shap_values, X_test_selected_df, feature_names=selected_feature_names)
plt.show()
print("SHAP summary plot generated.")

In [ ]:
pip install lime

### 1. LIME (Local Interpretable Model-agnostic Explanations)

LIME is a technique that explains the predictions of any machine learning classifier by approximating it locally with an interpretable model. It works by perturbing the input data, getting predictions from the black-box model, and then fitting a simpler, interpretable model (like linear regression or decision tree) on these perturbed samples weighted by their proximity to the original instance. This local model then provides an explanation for the specific prediction of that instance.

Here, we'll use LIME to explain a single prediction from our `stacking_model_xgb_meta`.

In [ ]:
import lime
import lime.lime_tabular

# Create a LIME explainer
# feature_names are crucial for interpretable output
# training_data is used by LIME to understand feature distributions for perturbations
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_selected,
    feature_names=selected_feature_names,
    class_names=['No Heart Disease', 'Heart Disease'], # Assuming 0=No, 1=Yes
    mode='classification'
)

# Choose an instance from the test set to explain
instance_idx = 4  # Explain the 5th instance (index 4) in the test set
chosen_instance = X_test_selected[instance_idx]

# Get the explanation for the chosen instance
# We need to pass the model's predict_proba method
explanation = explainer.explain_instance(
    data_row=chosen_instance,
    predict_fn=stacking_model_xgb_meta.predict_proba,
    num_features=10 # Number of features to include in the explanation
)

print(f"LIME Explanation for instance {instance_idx}:")
print(f"Predicted class: {stacking_model_xgb_meta.predict([chosen_instance])[0]}")
print(f"Predicted probabilities: {stacking_model_xgb_meta.predict_proba([chosen_instance])}")

# Print the explanation as a list of (feature, weight) tuples
for feature, weight in explanation.as_list():
    print(f"  {feature}: {weight:.4f}")

In [ ]:
# Visualize the LIME explanation
print("Generating LIME explanation plot...")
explanation.show_in_notebook(show_table=True, show_all=False)
print("LIME explanation plot generated.")